# Compensation relationships
**Jamal Ford — Part B — notebook 03**

**Dataset status:** Uses Samia's cleaned GitHub dataset with **4,435 respondents**. The source is `data/processed/us_fulltime_clean.csv`, prepared in notebook 01 at GitHub commit `02b883a`. This is the current team-data analysis.

This notebook compares compensation by professional experience, role family, and work
arrangement, and checks whether the work-arrangement and role gaps remain within
experience bands. It loads the team processed CSV and runs independently of notebook 02.
No additional observations are removed from the prepared sample.

Groups and experience boundaries are inherited from notebook 01. The original
`20+` band is displayed as **21+** because its members have more than 20 years of
experience. `log_pay` is the natural logarithm. Missing categories are kept visible.

The n>=30 screen used for conditional plots is a descriptive stability rule, not
a significance test or confidence interval. The tables retain every observed cell.
No comparison here establishes that remote work, a role, or experience causes pay.

**Shared helper:** [eda_common.py](../scripts/eda_common.py) contains repeated input checks, summaries, and chart formatting. Keep the scripts and data folders with these notebooks. In VS Code, open the whole project folder, restart the kernel, and run all cells.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown

# Locate the team files from the notebook folder, project folder, or workspace root.
ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for folder in [candidate, candidate / "Project/Developer_Compensation",
                   candidate / "Developer_Compensation", candidate / "Project"]:
        if (folder / "scripts/eda_common.py").exists() and (folder / "data/docs/team_source.json").exists():
            ROOT = folder
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the project folder with its data and scripts.")
sys.path.insert(0, str(ROOT / "scripts"))
import importlib
import eda_common
importlib.reload(eda_common)
from eda_common import (
    load_data, pay_summary, band_order, count_plot, box_plot, dollars,
    wrapped, save_figure, MIN_GROUP_N, MISSING,
)
df, info, output = load_data(ROOT)
pd.set_option("display.max_columns", 30)
display(Markdown(f"**Prepared team sample: {len(df):,} U.S. employed respondents.** "
                 "All input rows are retained in this EDA."))
display(pd.Series(info, name="Run information").to_frame())


## Pay by experience role and work arrangement
Each table reports n, median, Q1, Q3, and IQR in dollars. Missing experience and work
arrangement are shown separately; they are not interpreted as known categories.
Every marginal table totals the complete prepared sample.


In [ ]:
tables = {}
for column, filename in [("ExperienceBand", "pay_by_experience"),
                         ("RoleGroup", "pay_by_role"),
                         ("WorkArrangement", "pay_by_work_arrangement")]:
    table = pay_summary(df, column)
    if column == "ExperienceBand":
        table = table.set_index(column).reindex(band_order(df)).reset_index()
    else:
        table = table.sort_values("median_usd", ascending=False)
    table.to_csv(output["tables"] / f"{filename}.csv", index=False)
    tables[column] = table
    display(Markdown(f"### Compensation by {column}"))
    display(table.round(2))


## Pay comparisons as box plots
Boxes show Q1 to Q3 and the median. Whiskers follow the 1.5-IQR convention; points
beyond the whiskers remain visible and remain in the summaries. All plots use the
annual dollar scale and show every retained pay observation, including those above
the box-plot whiskers. Notebook 01's pay cutoffs have already been applied.


In [ ]:
role_order = tables["RoleGroup"].sort_values("median_usd")["RoleGroup"].tolist()
work_order = [x for x in ["In-person", "Hybrid", "Remote", MISSING] if x in set(df.WorkArrangement)]
box_plot(df, "ExperienceBand", band_order(df), "Compensation by professional work experience",
         "pay_by_experience", info, output)
box_plot(df, "RoleGroup", role_order, "Compensation by prepared role family",
         "pay_by_role", info, output)
box_plot(df, "WorkArrangement", work_order, "Compensation by prepared work arrangement",
         "pay_by_remote", info, output)


## Professional experience versus natural log compensation
The scatterplot uses complete experience/pay pairs. Pearson correlation describes
the linear association with ln pay. Spearman correlation compares ranks and can
describe a monotonic association even if the trend is not linear. These statistics
are descriptive and do not adjust for role or work arrangement.


In [ ]:
complete = df.dropna(subset=["ExperienceYears", "LogPay"])
pearson = complete.ExperienceYears.corr(complete.LogPay)
spearman = complete.ExperienceYears.rank().corr(complete.PayUSD.rank())
correlations = pd.DataFrame([{"complete_pairs_n": len(complete),
    "pearson_experience_ln_pay": pearson, "spearman_experience_pay": spearman}])
correlations.to_csv(output["tables"] / "experience_pay_correlations.csv", index=False)
display(correlations.round(3))
fig, ax = plt.subplots(figsize=(10, 5))
colors = {"Remote": "#31688e", "Hybrid": "#35a28c", "In-person": "#c86b31", MISSING: "#888888"}
for label, color in colors.items():
    subset = complete.loc[complete.WorkArrangement.eq(label)]
    if len(subset):
        ax.scatter(subset.ExperienceYears, subset.LogPay, s=14, alpha=.25,
                   edgecolors="none", color=color, label=f"{label} (n={len(subset):,})")
ax.set(title=f"Experience and natural-log compensation (complete pairs n={len(complete):,})",
       xlabel="Years of professional work experience", ylabel="ln annual total compensation in USD")
ax.legend(fontsize=9, loc="lower right")
ax.grid(alpha=.15)
save_figure(fig, "exp_vs_logpay", info, output)


## Work arrangement within experience bands
The arrangement-by-experience table includes median/IQR/n for every observed cell.
The composition table shows within-band arrangement proportions, including missing
arrangements in the denominator. Conditional pay comparisons exclude missing experience.

Remote-minus-in-person gaps compare the two exact prepared categories. Hybrid stays
separate. The headline comparison and gap range require at least 30 respondents in
both categories. The plot also displays Hybrid medians wherever that cell has n>=30.


In [ ]:
bands = [band for band in band_order(df) if band != MISSING]
known_exp = df.loc[df.ExperienceBand.ne(MISSING)]
work_experience = pay_summary(known_exp, ["ExperienceBand", "WorkArrangement"])
work_experience["n_at_least_30"] = work_experience["n"].ge(MIN_GROUP_N)
work_experience.to_csv(output["tables"] / "pay_by_work_and_experience.csv", index=False)
composition = pd.crosstab(known_exp.ExperienceBand, known_exp.WorkArrangement, normalize="index").reindex(bands)
composition.to_csv(output["tables"] / "work_arrangement_proportions_within_experience.csv")
display(work_experience.round(2))
display(composition.round(3))
records = []
for band in bands:
    group = known_exp.loc[known_exp.ExperienceBand.eq(band)]
    remote = group.loc[group.WorkArrangement.eq("Remote"), "PayUSD"]
    inperson = group.loc[group.WorkArrangement.eq("In-person"), "PayUSD"]
    r, p = remote.median(), inperson.median()
    records.append({"ExperienceBand": band, "remote_n": len(remote), "inperson_n": len(inperson),
        "remote_median_usd": r, "inperson_median_usd": p, "gap_usd": r-p,
        "gap_pct": 100*(r/p-1) if pd.notna(p) and p>0 else np.nan,
        "both_groups_n_at_least_30": len(remote)>=MIN_GROUP_N and len(inperson)>=MIN_GROUP_N})
remote_by_band = pd.DataFrame(records)
remote_by_band.to_csv(output["tables"] / "remote_inperson_within_experience.csv", index=False)
display(remote_by_band.round(2))
eligible = remote_by_band.loc[remote_by_band.both_groups_n_at_least_30]
fig, ax = plt.subplots(figsize=(11, 5.6))
for arrangement, color, offset in [("Remote", "#31688e", 10), ("Hybrid", "#35a28c", 10),
                                    ("In-person", "#c86b31", -16)]:
    rows = work_experience.loc[work_experience.WorkArrangement.eq(arrangement)].set_index("ExperienceBand").reindex(bands)
    medians = rows.median_usd.where(rows["n"].ge(MIN_GROUP_N))
    ax.plot(range(len(bands)), medians, "o-", label=arrangement, color=color)
    for i, band in enumerate(bands):
        if pd.notna(medians.loc[band]):
            ax.annotate(f"n={int(rows.loc[band, 'n'])}", (i, medians.loc[band]),
                        xytext=(0, offset), textcoords="offset points", ha="center", fontsize=8, color=color)
ax.set_xticks(range(len(bands)), bands)
ax.set(title="Work-arrangement medians within experience bands (shown cells n >= 30)",
       xlabel="Professional work experience band in years", ylabel="Median annual total compensation in USD")
ax.yaxis.set_major_formatter(FuncFormatter(dollars))
ax.margins(y=.19)
ax.legend()
ax.grid(axis="y", alpha=.2)
save_figure(fig, "pay_remote_within_experience", info, output)


## Role family within experience bands
All nine prepared role families are compared within the same experience bands.
Each heatmap cell shows its median and n; cells below n=30 are masked. The CSV
retains all medians, quartiles, and counts, including the small cells.

We also compare the two most common named role families, chosen by sample size
rather than compensation. This checks a specific pooled role comparison within
experience bands without selecting pairs based on the largest observed gap.


In [ ]:
role_experience = pay_summary(known_exp, ["RoleGroup", "ExperienceBand"])
role_experience["n_at_least_30"] = role_experience["n"].ge(MIN_GROUP_N)
role_experience.to_csv(output["tables"] / "pay_by_role_and_experience.csv", index=False)
display(role_experience.round(2))
medians = role_experience.pivot(index="RoleGroup", columns="ExperienceBand", values="median_usd").reindex(index=role_order, columns=bands)
counts = role_experience.pivot(index="RoleGroup", columns="ExperienceBand", values="n").reindex(index=role_order, columns=bands).fillna(0)
visible = medians.where(counts.ge(MIN_GROUP_N))
fig, ax = plt.subplots(figsize=(11, 8))
cmap = plt.colormaps["Blues"].copy()
cmap.set_bad("#eeeeee")
heat = ax.imshow(np.ma.masked_invalid(visible.to_numpy(dtype=float)), aspect="auto", cmap=cmap)
colorbar = fig.colorbar(heat, ax=ax, fraction=.035, pad=.03)
colorbar.ax.yaxis.set_major_formatter(FuncFormatter(dollars))
colorbar.set_label("Median annual total compensation in USD")
ax.set_xticks(range(len(bands)), bands)
ax.set_yticks(range(len(visible)), wrapped(visible.index, 28))
threshold = (np.nanmin(visible.to_numpy())+np.nanmax(visible.to_numpy()))/2
for i in range(len(visible)):
    for j in range(len(bands)):
        value, n = visible.iloc[i,j], int(counts.iloc[i,j])
        label = f"${value/1000:,.0f}k\nn={n}" if pd.notna(value) else f"n={n}\nnot shown"
        ax.text(j, i, label, ha="center", va="center", fontsize=9,
                color="white" if pd.notna(value) and value>threshold else "#222222")
ax.set(title="Role medians within experience bands (shown cells n >= 30)",
       xlabel="Professional work experience band in years")
save_figure(fig, "pay_role_within_experience", info, output)
named = df.loc[~df.RoleGroup.isin(["Other", MISSING]), "RoleGroup"].value_counts().index[:2].tolist()
pair_records = []
if len(named)==2:
    reference, comparison = named
    for band in bands:
        group = known_exp.loc[known_exp.ExperienceBand.eq(band)]
        a = group.loc[group.RoleGroup.eq(reference), "PayUSD"]
        b = group.loc[group.RoleGroup.eq(comparison), "PayUSD"]
        pair_records.append({"ExperienceBand": band, "reference_role": reference,
            "comparison_role": comparison, "reference_n": len(a), "comparison_n": len(b),
            "reference_median_usd": a.median(), "comparison_median_usd": b.median(),
            "comparison_minus_reference_usd": b.median()-a.median(),
            "both_groups_n_at_least_30": len(a)>=MIN_GROUP_N and len(b)>=MIN_GROUP_N})
role_pair = pd.DataFrame(pair_records)
role_pair.to_csv(output["tables"] / "common_role_pair_within_experience.csv", index=False)
display(role_pair.round(2))


## Additional check of role composition in the remote comparison
Remote and In-person medians are also compared within each role-and-experience
cell. Both categories need n>=30 for an eligible comparison. This checks more of
the observed group composition, but does not account for employer, detailed
seniority, hours, location within the United States, or other unmeasured factors.


In [ ]:
three_way_records = []
for (role, band), group in known_exp.groupby(["RoleGroup", "ExperienceBand"], observed=True):
    r = group.loc[group.WorkArrangement.eq("Remote"), "PayUSD"]
    p = group.loc[group.WorkArrangement.eq("In-person"), "PayUSD"]
    three_way_records.append({"RoleGroup": role, "ExperienceBand": band,
        "remote_n": len(r), "inperson_n": len(p), "remote_median_usd": r.median(),
        "inperson_median_usd": p.median(), "gap_usd": r.median()-p.median(),
        "both_groups_n_at_least_30": len(r)>=MIN_GROUP_N and len(p)>=MIN_GROUP_N})
three_way = pd.DataFrame(three_way_records)
three_way.to_csv(output["tables"] / "remote_inperson_within_role_and_experience.csv", index=False)
display(three_way.loc[three_way.both_groups_n_at_least_30].round(2))


## Findings and limitations for the report
The draft below is calculated from this run and written to `report/jamal_eda_findings.md`.
It covers the EDA and Findings sections and Jamal's limitations contribution. It
does not replace the data-preparation, modeling, or recommendations sections.


In [ ]:
overall = pay_summary(df).iloc[0]
work = tables["WorkArrangement"].set_index("WorkArrangement")
remote, inperson = work.loc["Remote"], work.loc["In-person"]
gap = remote.median_usd-inperson.median_usd
work_text = (
    f"Remote respondents have median annual compensation {dollars(remote.median_usd)} "
    f"(n={int(remote.n):,}), compared with {dollars(inperson.median_usd)} for In-person "
    f"respondents (n={int(inperson.n):,}). The pooled remote-minus-in-person median gap is "
    f"{dollars(gap)} ({100*gap/inperson.median_usd:.1f}% of the in-person median). "
)
if len(eligible):
    positive = int(eligible.gap_usd.gt(0).sum())
    work_text += (f"Among {len(eligible)} experience bands with both categories n>=30, "
        f"{positive} have a positive remote-minus-in-person gap. The within-band gaps range "
        f"from {dollars(eligible.gap_usd.min())} to {dollars(eligible.gap_usd.max())}. "
        "The size and direction should be read from the within-band table rather than "
        "assumed to equal the pooled difference. ")
role_table = tables["RoleGroup"].loc[~tables["RoleGroup"].RoleGroup.isin(["Other", MISSING])]
top = role_table.sort_values("median_usd", ascending=False).iloc[0]
bottom = role_table.sort_values("median_usd").iloc[0]
role_text = (f"Among named role families, {top.RoleGroup} has the highest pooled median "
    f"({dollars(top.median_usd)}, n={int(top.n):,}), while {bottom.RoleGroup} has the lowest "
    f"({dollars(bottom.median_usd)}, n={int(bottom.n):,}). These families combine different jobs. ")
pair_ok = role_pair.loc[role_pair.both_groups_n_at_least_30]
if len(pair_ok):
    first = pair_ok.iloc[0]
    positive = int(pair_ok.comparison_minus_reference_usd.gt(0).sum())
    role_text += (f"The two most common named families are {first.reference_role} and "
        f"{first.comparison_role}. Of {len(pair_ok)} experience bands with both families "
        f"n>=30, {positive} have a higher median for {first.comparison_role}; the "
        f"comparison-minus-reference gap ranges from {dollars(pair_ok.comparison_minus_reference_usd.min())} "
        f"to {dollars(pair_ok.comparison_minus_reference_usd.max())}. ")
three_ok = three_way.loc[three_way.both_groups_n_at_least_30]
three_text = (f"In the additional role-by-experience check, {len(three_ok)} cells have at least "
    f"30 Remote and 30 In-person respondents; {int(three_ok.gap_usd.gt(0).sum())} of those "
    "have a positive remote-minus-in-person median gap. Cells excluded by the screen "
    "are still listed with counts in the CSV. This check does not establish a causal effect. ")
findings = (
    "# Jamal Part B EDA and findings\n\n"
    "## Sample and scope\n\n"
    f"This analysis uses all {len(df):,} rows of the team's prepared CSV. The preparation "
    "notebook selects U.S. employed respondents who report compensation and keeps pay "
    "between $10,000 and the reporters' 99th percentile ($641,900), excluding 97 of "
    "4,532 pay reporters. No further respondents are removed in this EDA. The 2025 "
    "survey does not directly identify full-time status, and notebook 01 does not apply "
    "a professional-developer identity filter. Findings describe this prepared sample.\n\n"
    "## Distributions\n\n"
    f"Median annual total compensation is {dollars(overall.median_usd)}; Q1 is "
    f"{dollars(overall.q1_usd)}, Q3 is {dollars(overall.q3_usd)}, and IQR is "
    f"{dollars(overall.iqr_usd)}. Mean pay is {dollars(df.PayUSD.mean())}, consistent with "
    "the remaining right tail. Raw and natural-log histograms show the same respondents. "
    f"Reported professional work experience has a median of {df.ExperienceYears.median():g} "
    f"years, with {info['missing_experience']} missing values. The nine role families, "
    "prepared work arrangements, original work options, education, and experience bands "
    "are shown in the categorical bar charts.\n\n"
    "## Experience and role relationships\n\n"
    f"The experience/ln-pay Pearson correlation is {pearson:.3f}; the rank correlation "
    f"between experience and pay is {spearman:.3f} (complete pairs n={len(complete):,}). "
    "These summarize different aspects of the association and are not a causal estimate. "
    + role_text + "The role-by-experience heatmap compares all nine families and masks "
    "cells below n=30.\n\n"
    "## Work arrangement within experience groups\n\n" + work_text + three_text + "\n\n"
    "## Interpretation and limitations\n\n"
    "The pooled pay gaps combine differences in the experience and role composition of "
    "the groups. Within-experience and within-role-and-experience comparisons provide "
    "more comparable descriptive groups, but broad bands leave residual experience "
    "differences. Hours, employer, seniority, specialization, and geography within the "
    "United States are not controlled. The results do not show that remote work or "
    "changing roles causes higher pay.\n\n"
    "Recruitment favors engaged Stack Overflow users and compensation reporting is "
    "optional. Notebook 01 found 881 non-reporters among 5,413 employed U.S. respondents; "
    "non-reporters had higher median reported experience and were less often Remote "
    "than reporters. The reporting sample can therefore differ from the broader "
    "respondent population. These findings should not be generalized to all U.S. workers.\n\n"
    "The pay cutoffs alter the tails and may exclude some genuine observations; "
    "sensitivity to alternative preparation cutoffs was not assessed by these notebooks. "
    "Professional work experience is broader than professional coding experience. "
    "Blank experience can mean zero years or nonresponse, so missing experience is "
    "kept visible and excluded from within-experience comparisons rather than imputed. "
    f"Work arrangement is missing for {info['missing_work_arrangement']} respondents "
    f"and education for {info['missing_education']}.\n\n"
    "Role families combine occupations, especially Other. The prepared Hybrid category "
    "includes Your choice; the detailed-work bar chart shows the original options. "
    "The original 20+ experience label is displayed as 21+ because notebook 01's "
    "boundaries place exactly 20 years in 11-20. Sparse conditional cells are flagged. "
    "The n>=30 display screen does not measure uncertainty or establish statistical "
    "significance.\n\n"
    "## Figures and tables\n\n"
    "- Distributions: pay_distribution.png, experience_distribution.png, and counts_*.png in figures/.\n"
    "- Group comparisons: pay_by_experience.png, pay_by_role.png, and pay_by_remote.png.\n"
    "- Conditional comparisons: pay_remote_within_experience.png and pay_role_within_experience.png.\n"
    "- Scatterplot: exp_vs_logpay.png. All dollar summaries and counts are exported to tables/.\n\n"
    "## Sources and reproducibility\n\n"
    "- [Team preparation notebook](../notebooks/01_data_prep.ipynb) and processed CSV.\n"
    "- [2025 survey archive](https://github.com/StackExchange/Survey/tree/main/packages/archive/2025).\n"
    "- [2025 work results](https://survey.stackoverflow.co/2025/work).\n"
    "- [2025 methodology](https://survey.stackoverflow.co/2025/methodology).\n"
    f"- Input SHA-256: `{info['input_sha256']}`. Full run information is in jamal_eda_run_metadata.json.\n"
)
(output["report"] / "jamal_eda_findings.md").write_text(findings, encoding="utf-8")
display(Markdown(findings))
